# 02 · Procesamiento: destinos por región, precios e indicadores

Este notebook toma los extractos del notebook 01 y prepara:

1. **Producto y tipo de producto** de cada subpartida (café verde o tostado; cacao en grano, manteca, chocolate, etc.).
2. **País y región de destino**: código ISO-3 (para el mapa), nombre en español y región: Unión Europea, resto de Europa, Norteamérica,
   América Latina y el Caribe, Asia y Medio Oriente, Oceanía y África.
3. **Indicadores anuales por producto**: valor FOB, volumen, valor unitario, precio internacional, descomposición del cambio del valor
   en *efecto precio* y *efecto volumen*, e indicadores de diversificación de mercados.
4. Los archivos del dashboard (`web/data/datos.json`).

**Periodo.** El análisis cubre 2015–2025 (años completos). 2026 solo tiene enero–agosto: se compara con enero–agosto de 2025 y se
muestra aparte, como señal de lo que está pasando ahora.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd()
ORIGINALES = RAIZ / "datos" / "originales"
PROCESADOS = RAIZ / "datos" / "procesados"
WEB_DATA = RAIZ / "web" / "data"
PROCESADOS.mkdir(parents=True, exist_ok=True)
WEB_DATA.mkdir(parents=True, exist_ok=True)

ANIO_INICIO, ANIO_FIN = 2015, 2025
PRODUCTOS = ["Café", "Cacao"]

g8 = pd.read_csv(ORIGINALES / "extracto_g8_cafe_cacao_mensual.csv", dtype={"subpartida": str})
precios = pd.read_csv(ORIGINALES / "extracto_precios_internacionales_mensual.csv")
metadatos = json.loads((ORIGINALES / "metadatos_descarga.json").read_text(encoding="utf-8"))
print(g8.shape, precios.shape)

(34199, 8) (801, 5)


## 1. Control de calidad

In [2]:
print("Duplicados por clave (año, mes, país, subpartida):", g8.duplicated(["anio", "mes", "pais_sunat", "subpartida"]).sum())
print("Valores negativos:", (g8[["fob_usd", "peso_neto_kg"]] < 0).sum().to_dict())
print("Filas con FOB > 0 y peso neto = 0:", ((g8.fob_usd > 0) & (g8.peso_neto_kg == 0)).sum())
ANIO_PARCIAL = int(g8.anio.max())
ULTIMO_MES = int(g8.loc[g8.anio == ANIO_PARCIAL, "mes"].max())
print(f"Último dato de exportaciones: {ANIO_PARCIAL}, mes {ULTIMO_MES}")
ultimo_precio = precios.dropna(subset=["cacao"]).iloc[-1]
print(f"Último dato de precios: {int(ultimo_precio.anio)}, mes {int(ultimo_precio.mes)}")

Duplicados por clave (año, mes, país, subpartida): 0
Valores negativos: {'fob_usd': 0, 'peso_neto_kg': 0}
Filas con FOB > 0 y peso neto = 0: 0
Último dato de exportaciones: 2026, mes 8
Último dato de precios: 2026, mes 9


## 2. Países y regiones de destino

SUNAT usa nombres propios para los países. Se arma una tabla de equivalencias a mano (la misma de la tarea anterior):
- Los departamentos franceses de ultramar se suman a **Francia**, "Escocia" al **Reino Unido** y "Serbia y Montenegro" a **Serbia**.
- Los destinos sin país (no declarados, zonas francas del Perú) llevan un código que empieza con `X` y no van al mapa.

Luego cada país se asigna a una **región**. La Unión Europea se define con sus **27 miembros actuales** en todos los años, para que la
serie sea comparable; el Reino Unido queda en "Resto de Europa" también antes del Brexit. México va en América Latina y el Caribe.

In [3]:
PAISES = {
    "AFGANISTAN": ("AFG", "Afganistán"), "AGUAS INTERNACIONALES": ("XAI", "Aguas internacionales"), "ALBANIA": ("ALB", "Albania"),
    "ALEMANIA": ("DEU", "Alemania"), "ANDORRA": ("AND", "Andorra"), "ANGOLA": ("AGO", "Angola"), "ANGUILLA": ("AIA", "Anguila"),
    "ANTIGUA Y BARBUDA": ("ATG", "Antigua y Barbuda"), "ANTILLAS HOLANDESAS": ("XAN", "Antillas Neerlandesas (extinta)"),
    "ARABIA SAUDITA": ("SAU", "Arabia Saudita"), "ARGELIA": ("DZA", "Argelia"), "ARGENTINA": ("ARG", "Argentina"),
    "ARMENIA": ("ARM", "Armenia"), "ARUBA": ("ABW", "Aruba"), "AUSTRALIA": ("AUS", "Australia"), "AUSTRIA": ("AUT", "Austria"),
    "AZERBAIJAN": ("AZE", "Azerbaiyán"), "BAHAMAS": ("BHS", "Bahamas"), "BAHREIN": ("BHR", "Baréin"), "BANGLADESH": ("BGD", "Bangladés"),
    "BARBADOS": ("BRB", "Barbados"), "BELARUS": ("BLR", "Bielorrusia"), "BELGICA": ("BEL", "Bélgica"), "BELICE": ("BLZ", "Belice"),
    "BENIN": ("BEN", "Benín"), "BERMUDAS": ("BMU", "Bermudas"), "BOLIVIA": ("BOL", "Bolivia"),
    "BONAIRE,SAN EUSTAQUIO Y SABA": ("BES", "Bonaire, San Eustaquio y Saba"), "BOSNIA-HERZEGOVINA": ("BIH", "Bosnia y Herzegovina"),
    "BOTSWANA": ("BWA", "Botsuana"), "BRASIL": ("BRA", "Brasil"), "BRUNEI DARUSSALAM": ("BRN", "Brunéi"), "BULGARIA": ("BGR", "Bulgaria"),
    "BURKINA FASO": ("BFA", "Burkina Faso"), "BURUNDI": ("BDI", "Burundi"), "BUTAN": ("BTN", "Bután"), "CABO VERDE": ("CPV", "Cabo Verde"),
    "CAIMAN,ISLAS": ("CYM", "Islas Caimán"), "CAMBOYA": ("KHM", "Camboya"), "CAMERUN,REPUBLICA UNIDA DEL": ("CMR", "Camerún"),
    "CANADA": ("CAN", "Canadá"), "CHAD": ("TCD", "Chad"), "CHILE": ("CHL", "Chile"), "CHINA": ("CHN", "China"), "CHIPRE": ("CYP", "Chipre"),
    "COCOS (KEELING),ISLAS": ("CCK", "Islas Cocos"), "COLOMBIA": ("COL", "Colombia"), "COMORAS": ("COM", "Comoras"),
    "COOK, ISLAS": ("COK", "Islas Cook"), "COREA (NORTE), REPUBLICA POPULAR DEMOCRATICA DE": ("PRK", "Corea del Norte"),
    "COREA (SUR), REPUBLICA DE": ("KOR", "Corea del Sur"), "COSTA DE MARFIL": ("CIV", "Costa de Marfil"), "COSTA RICA": ("CRI", "Costa Rica"),
    "COTE D'IVOIRE": ("CIV", "Costa de Marfil"), "CROACIA": ("HRV", "Croacia"), "CUBA": ("CUB", "Cuba"), "CURACAO": ("CUW", "Curazao"),
    "DINAMARCA": ("DNK", "Dinamarca"), "DJIBOUTI": ("DJI", "Yibuti"), "DOMINICA": ("DMA", "Dominica"), "ECUADOR": ("ECU", "Ecuador"),
    "EGIPTO": ("EGY", "Egipto"), "EL SALVADOR": ("SLV", "El Salvador"), "EMIRATOS ARABES UNIDOS": ("ARE", "Emiratos Árabes Unidos"),
    "ERITREA": ("ERI", "Eritrea"), "ESCOCIA": ("GBR", "Reino Unido"), "ESLOVAQUIA": ("SVK", "Eslovaquia"), "ESLOVENIA": ("SVN", "Eslovenia"),
    "ESPAÑA": ("ESP", "España"), "ESTADOS UNIDOS": ("USA", "Estados Unidos"), "ESTONIA": ("EST", "Estonia"), "ETIOPIA": ("ETH", "Etiopía"),
    "FEROE, ISLAS": ("FRO", "Islas Feroe"), "FIJI": ("FJI", "Fiyi"), "FILIPINAS": ("PHL", "Filipinas"), "FINLANDIA": ("FIN", "Finlandia"),
    "FRANCE, METROPOLITAN": ("FRA", "Francia"), "FRANCIA": ("FRA", "Francia"), "GABON": ("GAB", "Gabón"), "GAMBIA": ("GMB", "Gambia"),
    "GEORGIA": ("GEO", "Georgia"), "GHANA": ("GHA", "Ghana"), "GIBRALTAR": ("GIB", "Gibraltar"), "GRANADA": ("GRD", "Granada"),
    "GRECIA": ("GRC", "Grecia"), "GROENLANDIA": ("GRL", "Groenlandia"), "GUADALUPE": ("FRA", "Francia"), "GUAM": ("GUM", "Guam"),
    "GUATEMALA": ("GTM", "Guatemala"), "GUAYANA FRANCESA": ("FRA", "Francia"), "GUINEA": ("GIN", "Guinea"),
    "GUINEA ECUATORIAL": ("GNQ", "Guinea Ecuatorial"), "GUINEA-BISSAU": ("GNB", "Guinea-Bisáu"), "GUYANA": ("GUY", "Guyana"),
    "HAITI": ("HTI", "Haití"), "HONDURAS": ("HND", "Honduras"), "HONG KONG": ("HKG", "Hong Kong"), "HUNGRIA": ("HUN", "Hungría"),
    "INDIA": ("IND", "India"), "INDONESIA": ("IDN", "Indonesia"), "IRAK": ("IRQ", "Irak"), "IRAN, REPUBLICA ISLAMICA DEL": ("IRN", "Irán"),
    "IRLANDA (EIRE)": ("IRL", "Irlanda"), "ISLANDIA": ("ISL", "Islandia"), "ISRAEL": ("ISR", "Israel"), "ITALIA": ("ITA", "Italia"),
    "JAMAICA": ("JAM", "Jamaica"), "JAPON": ("JPN", "Japón"), "JORDANIA": ("JOR", "Jordania"), "KAZAJSTAN": ("KAZ", "Kazajistán"),
    "KENIA": ("KEN", "Kenia"), "KIRGUIZISTAN": ("KGZ", "Kirguistán"), "KIRIBATI": ("KIR", "Kiribati"), "KUWAIT": ("KWT", "Kuwait"),
    "LAOS, REPUBLICA POPULAR DEMOCRATICA DE": ("LAO", "Laos"), "LETONIA": ("LVA", "Letonia"), "LIBANO": ("LBN", "Líbano"),
    "LIBERIA": ("LBR", "Liberia"), "LIBIA": ("LBY", "Libia"), "LIECHTENSTEIN": ("LIE", "Liechtenstein"), "LITUANIA": ("LTU", "Lituania"),
    "LUXEMBURGO": ("LUX", "Luxemburgo"), "MACAO": ("MAC", "Macao"), "MACEDONIA F.Y.R.": ("MKD", "Macedonia del Norte"),
    "MADAGASCAR": ("MDG", "Madagascar"), "MALAWI": ("MWI", "Malaui"), "MALAYSIA": ("MYS", "Malasia"), "MALDIVAS": ("MDV", "Maldivas"),
    "MALI": ("MLI", "Malí"), "MALTA": ("MLT", "Malta"), "MARIANAS DEL NORTE, ISLAS": ("MNP", "Islas Marianas del Norte"),
    "MARRUECOS": ("MAR", "Marruecos"), "MARSHALL, ISLAS": ("MHL", "Islas Marshall"), "MARTINICA": ("FRA", "Francia"),
    "MAURICIO": ("MUS", "Mauricio"), "MAURITANIA": ("MRT", "Mauritania"), "MAYOTTE": ("FRA", "Francia"), "MEXICO": ("MEX", "México"),
    "MICRONESIA, ESTADOS FEDERADOS DE": ("FSM", "Micronesia"), "MOLDAVIA": ("MDA", "Moldavia"), "MONACO": ("MCO", "Mónaco"),
    "MONGOLIA": ("MNG", "Mongolia"), "MONTENEGRO": ("MNE", "Montenegro"), "MONTSERRAT, ISLA": ("MSR", "Montserrat"),
    "MOZAMBIQUE": ("MOZ", "Mozambique"), "MYANMAR": ("MMR", "Myanmar"), "NAMIBIA": ("NAM", "Namibia"), "NAURU": ("NRU", "Nauru"),
    "NAVIDAD (CHRISTMAS), ISLA": ("CXR", "Isla de Navidad"), "NEPAL": ("NPL", "Nepal"), "NICARAGUA": ("NIC", "Nicaragua"),
    "NIGER": ("NER", "Níger"), "NIGERIA": ("NGA", "Nigeria"), "NO DECLARADOS": ("XND", "No declarado"), "NORFOLK, ISLA": ("NFK", "Isla Norfolk"),
    "NORUEGA": ("NOR", "Noruega"), "NUEVA CALEDONIA": ("NCL", "Nueva Caledonia"), "NUEVA ZELANDA": ("NZL", "Nueva Zelanda"),
    "OMAN": ("OMN", "Omán"), "PAISES BAJOS": ("NLD", "Países Bajos"),
    "PAISES NO PRECISADOS EN OTRA PARTE Y DESCONOCIDOS": ("XNP", "País no precisado"), "PAKISTAN": ("PAK", "Pakistán"),
    "PALAU, ISLAS": ("PLW", "Palaos"), "PANAMA": ("PAN", "Panamá"), "PAPUASIA NUEVA GUINEA": ("PNG", "Papúa Nueva Guinea"),
    "PARAGUAY": ("PRY", "Paraguay"), "PERÚ": ("XPE", "Perú (destino nacional)"), "POLINESIA FRANCESA": ("PYF", "Polinesia Francesa"),
    "POLONIA": ("POL", "Polonia"), "PORTUGAL": ("PRT", "Portugal"), "PUERTO RICO": ("PRI", "Puerto Rico"), "QATAR": ("QAT", "Catar"),
    "REINO UNIDO": ("GBR", "Reino Unido"), "REPUBLICA CENTROAFRICANA": ("CAF", "República Centroafricana"),
    "REPUBLICA CHECA": ("CZE", "Chequia"), "REPUBLICA DOMINICANA": ("DOM", "República Dominicana"),
    "REPÚBLICA DEL CONGO": ("COG", "República del Congo"), "REPÚBLICA DEMOCRÁTICA DEL CONGO": ("COD", "R. D. del Congo"),
    "REUNION": ("FRA", "Francia"), "RUANDA": ("RWA", "Ruanda"), "RUMANIA": ("ROU", "Rumania"), "RUSIA": ("RUS", "Rusia"),
    "SALOMON, ISLAS": ("SLB", "Islas Salomón"), "SAMOA": ("WSM", "Samoa"), "SAMOA NORTEAMERICANA": ("ASM", "Samoa Americana"),
    "SAN BARTOLOME": ("BLM", "San Bartolomé"), "SAN CRISTOBAL Y NIEVES": ("KNA", "San Cristóbal y Nieves"),
    "SAN MARINO": ("SMR", "San Marino"), "SAN MARTIN": ("MAF", "San Martín"),
    "SAN VICENTE Y LAS GRANADINAS": ("VCT", "San Vicente y las Granadinas"), "SANTA ELENA": ("SHN", "Santa Elena"),
    "SANTA LUCIA": ("LCA", "Santa Lucía"), "SANTA SEDE": ("VAT", "Santa Sede"), "SANTO TOME Y PRINCIPE": ("STP", "Santo Tomé y Príncipe"),
    "SAWSILANDIA": ("SWZ", "Esuatini"), "SENEGAL": ("SEN", "Senegal"), "SERBIA": ("SRB", "Serbia"), "SERBIA Y MONTENEGRO": ("SRB", "Serbia"),
    "SEYCHELLES": ("SYC", "Seychelles"), "SIERRA LEONA": ("SLE", "Sierra Leona"), "SINGAPUR": ("SGP", "Singapur"),
    "SIRIA, REPUBLICA ARABE DE": ("SYR", "Siria"), "SOMALIA": ("SOM", "Somalia"), "SRI LANKA": ("LKA", "Sri Lanka"),
    "SUDAFRICA, REPUBLICA DE": ("ZAF", "Sudáfrica"), "SUDAN": ("SDN", "Sudán"), "SUECIA": ("SWE", "Suecia"), "SUIZA": ("CHE", "Suiza"),
    "SURINAM": ("SUR", "Surinam"), "SVALBARD AND JAN MAYEN ISLANDS": ("SJM", "Svalbard y Jan Mayen"), "TAILANDIA": ("THA", "Tailandia"),
    "TAIWAN (FORMOSA)": ("TWN", "Taiwán"), "TANZANIA, REPUBLICA UNIDA DE": ("TZA", "Tanzania"),
    "TERRITORIO ANTARTICO BRITANICO": ("XAT", "Territorio Antártico Británico"),
    "TERRITORIO BRITANICO DEL OCEANO INDICO": ("IOT", "Territorio Británico del Océano Índico"), "TIMOR DEL ESTE": ("TLS", "Timor Oriental"),
    "TOGO": ("TGO", "Togo"), "TOKELAU": ("TKL", "Tokelau"), "TONGA": ("TON", "Tonga"), "TRINIDAD Y TOBAGO": ("TTO", "Trinidad y Tobago"),
    "TUNICIA": ("TUN", "Túnez"), "TURCAS Y CAICOS, ISLAS": ("TCA", "Islas Turcas y Caicos"), "TURKMENISTAN": ("TKM", "Turkmenistán"),
    "TURQUIA": ("TUR", "Turquía"), "UCRANIA": ("UKR", "Ucrania"), "UGANDA": ("UGA", "Uganda"),
    "UNITED STATES MINOR OUTLYING ISLANDS": ("UMI", "Islas menores de EE. UU."), "URUGUAY": ("URY", "Uruguay"),
    "UZBEKISTAN": ("UZB", "Uzbekistán"), "VANUATU": ("VUT", "Vanuatu"), "VENEZUELA": ("VEN", "Venezuela"), "VIET NAM": ("VNM", "Vietnam"),
    "VIRGENES, ISLAS (BRITANICAS)": ("VGB", "Islas Vírgenes Británicas"), "VIRGENES, ISLAS (NORTEAMERICANAS)": ("VIR", "Islas Vírgenes de EE. UU."),
    "YEMEN": ("YEM", "Yemen"), "YUGOSLAVIA": ("SRB", "Serbia"), "ZAMBIA": ("ZMB", "Zambia"), "ZIMBABWE": ("ZWE", "Zimbabue"),
    "ZONA NEUTRAL (PALESTINA)": ("PSE", "Palestina"), "ZONAS FRANCAS DEL PERU": ("XZF", "Zonas francas del Perú"),
}

REGIONES = {
    "Unión Europea": {
        "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
        "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE",
    },
    "Resto de Europa": {
        "ALB", "AND", "BLR", "BIH", "FRO", "GIB", "ISL", "LIE", "MKD", "MDA", "MCO", "MNE", "NOR", "RUS", "SMR", "VAT", "SRB",
        "CHE", "SJM", "TUR", "UKR", "GBR",
    },
    "Norteamérica": {"USA", "CAN", "BMU", "GRL"},
    "América Latina y el Caribe": {
        "AIA", "ATG", "ARG", "ABW", "BHS", "BRB", "BLZ", "BOL", "BES", "BRA", "CYM", "CHL", "COL", "CRI", "CUB", "CUW", "DMA", "DOM",
        "ECU", "SLV", "GRD", "GTM", "GUY", "HTI", "HND", "JAM", "MEX", "MSR", "NIC", "PAN", "PRY", "PRI", "KNA", "LCA", "VCT", "BLM",
        "MAF", "SUR", "TTO", "TCA", "URY", "VEN", "VGB", "VIR", "XAN",
    },
    "Asia y Medio Oriente": {
        "AFG", "SAU", "ARM", "AZE", "BHR", "BGD", "BTN", "BRN", "KHM", "CHN", "PRK", "KOR", "ARE", "GEO", "HKG", "IND", "IDN", "IRQ",
        "IRN", "ISR", "JPN", "JOR", "KAZ", "KGZ", "KWT", "LAO", "LBN", "MAC", "MYS", "MDV", "MNG", "MMR", "NPL", "OMN", "PAK", "PHL",
        "QAT", "SGP", "SYR", "LKA", "THA", "TWN", "TLS", "TKM", "UZB", "VNM", "YEM", "PSE", "IOT",
    },
    "Oceanía": {
        "AUS", "COK", "FJI", "KIR", "MHL", "FSM", "NRU", "NCL", "NZL", "NFK", "PLW", "PNG", "PYF", "SLB", "WSM", "ASM", "TKL", "TON",
        "VUT", "GUM", "MNP", "CCK", "CXR", "UMI",
    },
    "África": {
        "AGO", "DZA", "BEN", "BWA", "BFA", "BDI", "CPV", "CMR", "TCD", "COM", "CIV", "DJI", "EGY", "ERI", "ETH", "GAB", "GMB", "GHA",
        "GIN", "GNQ", "GNB", "KEN", "LBR", "LBY", "MDG", "MWI", "MLI", "MAR", "MUS", "MRT", "MOZ", "NAM", "NER", "NGA", "CAF", "COG",
        "COD", "RWA", "SHN", "STP", "SWZ", "SEN", "SYC", "SLE", "SOM", "ZAF", "SDN", "TZA", "TGO", "TUN", "UGA", "ZMB", "ZWE",
    },
}
SIN_PAIS = "Sin país declarado"
ORDEN_REGIONES = list(REGIONES) + [SIN_PAIS]

region_de = {}
for region, codigos in REGIONES.items():
    for iso in codigos:
        assert iso not in region_de, f"{iso} está en dos regiones"
        region_de[iso] = region

sin_mapear = sorted(set(g8.pais_sunat) - set(PAISES))
assert not sin_mapear, f"Países sin equivalencia: {sin_mapear}"
assert len(REGIONES["Unión Europea"]) == 27

tabla_paises = pd.DataFrame([(k, *v) for k, v in PAISES.items()], columns=["pais_sunat", "iso3", "pais"])
tabla_paises["region"] = tabla_paises.iso3.map(region_de).fillna(SIN_PAIS)
assert tabla_paises.query("region == @SIN_PAIS").iso3.str.startswith("X").all(), "Hay países reales sin región"
tabla_paises.to_csv(PROCESADOS / "tabla_paises.csv", index=False)
tabla_paises.drop_duplicates("iso3").region.value_counts()

region
África                        53
Asia y Medio Oriente          49
América Latina y el Caribe    45
Unión Europea                 27
Oceanía                       24
Resto de Europa               22
Sin país declarado             6
Norteamérica                   4
Name: count, dtype: int64

## 3. Producto y tipo de producto

El **producto base** de cada cadena es el que se compara con el precio internacional: **café verde** (0901.11, sin tostar ni
descafeinar) y **cacao en grano** (1801). Los demás tipos (tostado, manteca, pasta, chocolate) tienen más valor agregado.

In [4]:
SUB_CACAO = {
    "1801": "Cacao en grano", "1802": "Cáscara y residuos de cacao", "1803": "Pasta de cacao",
    "1804": "Manteca de cacao", "1805": "Cacao en polvo", "1806": "Chocolate y preparaciones",
}
BASE = {"Café": "Café verde", "Cacao": "Cacao en grano"}
PRECIO_REF = {"Café": "cafe_arabica", "Cacao": "cacao"}


def clasificar(codigo: str) -> tuple[str, str]:
    if codigo.startswith("0901"):
        if codigo.startswith("090111"):
            return "Café", "Café verde"
        if codigo.startswith("090112"):
            return "Café", "Café verde descafeinado"
        if codigo.startswith("09012"):
            return "Café", "Café tostado"
        return "Café", "Cáscara, cascarilla y sucedáneos"
    return "Cacao", SUB_CACAO[codigo[:4]]


clasif = pd.DataFrame([(c, *clasificar(c)) for c in g8.subpartida.unique()], columns=["subpartida", "producto", "subgrupo"])
datos = (
    g8.merge(clasif, on="subpartida")
    .merge(tabla_paises[["pais_sunat", "iso3", "pais", "region"]], on="pais_sunat", how="left")
)
assert datos.iso3.notna().all()

claves = ["anio", "mes", "producto", "subgrupo", "subpartida", "descripcion", "iso3", "pais", "region"]
mensual = (
    datos.groupby(claves, as_index=False)[["fob_usd", "peso_neto_kg"]].sum()
    .sort_values(["anio", "mes", "producto", "fob_usd"], ascending=[True, True, True, False])
)
mensual.to_csv(PROCESADOS / "exportaciones_mensual.csv", index=False)
print(mensual.shape)

composicion = (
    mensual.query("anio in [@ANIO_INICIO, @ANIO_FIN]").groupby(["producto", "subgrupo", "anio"]).fob_usd.sum()
    .unstack("anio").pipe(lambda t: t / t.groupby(level="producto").transform("sum") * 100).round(1)
)
composicion

(34187, 11)


anio                                       2015  2025
producto subgrupo                                    
Cacao    Cacao en grano                    72.1  60.1
         Cacao en polvo                     4.0   6.5
         Chocolate y preparaciones          5.8   6.1
         Cáscara y residuos de cacao        0.0   0.2
         Manteca de cacao                  16.1  20.4
         Pasta de cacao                     1.9   6.7
Café     Café tostado                       0.3   0.1
         Café verde                        99.2  99.8
         Café verde descafeinado            0.1   0.0
         Cáscara, cascarilla y sucedáneos   0.4   0.1

El producto base domina: el café verde es casi todo el café exportado y el cacao en grano es la mayor parte del cacao. Por eso su valor
unitario sigue de cerca al precio internacional.

## 4. Descomposición del cambio del valor: precio frente a volumen

El valor exportado es precio × cantidad. Para saber cuánto de su cambio entre dos periodos viene de cada parte se usa la fórmula de
**punto medio** (Marshall-Edgeworth), que suma exacto:

$$\Delta V = \underbrace{\sum_s \Delta p_s \cdot \tfrac{q_{s,0}+q_{s,1}}{2}}_{\text{efecto precio}}
 + \underbrace{\sum_s \Delta q_s \cdot \tfrac{p_{s,0}+p_{s,1}}{2}}_{\text{efecto volumen}}$$

donde $s$ es cada tipo de producto (café verde, tostado, cacao en grano, manteca…), $p$ el valor unitario (US$/kg) y $q$ el peso neto.
Calcularlo por tipo evita que un cambio en la mezcla (más chocolate y menos grano, por ejemplo) se confunda con un cambio de precio. Un
tipo que aparece o desaparece entre los dos periodos aporta solo al efecto volumen.

In [5]:
def descomponer(t0: pd.DataFrame, t1: pd.DataFrame) -> pd.Series:
    """Cambio del FOB entre dos periodos = efecto precio + efecto volumen. t0, t1: columnas fob y kg, índice = subgrupo."""
    t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
    p0 = (t.fob0 / t.kg0).where(t.kg0 > 0)
    p1 = (t.fob1 / t.kg1).where(t.kg1 > 0)
    p0, p1 = p0.fillna(p1), p1.fillna(p0)
    resultado = pd.Series({
        "cambio_fob": t.fob1.sum() - t.fob0.sum(),
        "efecto_precio": ((p1 - p0) * (t.kg0 + t.kg1) / 2).sum(),
        "efecto_volumen": ((t.kg1 - t.kg0) * (p0 + p1) / 2).sum(),
    })
    assert abs(resultado.efecto_precio + resultado.efecto_volumen - resultado.cambio_fob) < 1
    return resultado


PERIODOS = {
    "anual": lambda d: d[d.anio < ANIO_PARCIAL],         # años completos
    "ene_ago": lambda d: d[d.mes <= ULTIMO_MES],          # enero..último mes publicado, para comparar con el año parcial
}


def por_subgrupo(d: pd.DataFrame) -> pd.DataFrame:
    return (d.groupby(["producto", "anio", "subgrupo"])[["fob_usd", "peso_neto_kg"]].sum()
            .rename(columns={"fob_usd": "fob", "peso_neto_kg": "kg"}))

## 5. Diversificación de mercados

Para cada producto y año se mide qué tan concentradas están las ventas por país de destino (según valor FOB):

- **Índice Herfindahl-Hirschman (IHH):** suma de los cuadrados de las participaciones (en %). Va de casi 0 (muchos destinos pequeños) a
  10 000 (un solo destino). Referencia usual: menos de 1 500 es poco concentrado, 1 500–2 500 moderado y más de 2 500 alto.
- **Mercados equivalentes** = 10 000 / IHH: cuántos destinos del mismo tamaño darían la misma concentración. Es más fácil de comunicar.
- **Número de destinos**, **principal destino** y su participación, y la **participación de la Unión Europea**.

Los destinos sin país (códigos `X`) no entran en el cálculo.

In [6]:
def diversificacion(d: pd.DataFrame) -> pd.DataFrame:
    paises = d[~d.iso3.str.startswith("X")].groupby(["producto", "anio", "iso3"]).fob_usd.sum()
    paises = paises[paises > 0]
    part = paises / paises.groupby(level=["producto", "anio"]).transform("sum") * 100
    principal = part.groupby(level=["producto", "anio"]).idxmax().map(lambda clave: clave[2])
    ue = part[part.index.get_level_values("iso3").isin(REGIONES["Unión Europea"])].groupby(level=["producto", "anio"]).sum()
    return pd.DataFrame({
        "ihh": (part ** 2).groupby(level=["producto", "anio"]).sum(),
        "n_destinos": part.groupby(level=["producto", "anio"]).size(),
        "principal_iso3": principal,
        "principal_part": part.groupby(level=["producto", "anio"]).max(),
        "part_ue": ue,
    }).assign(mercados_equivalentes=lambda t: 10_000 / t.ihh)

## 6. Indicadores anuales por producto

Para cada producto, año y periodo (`anual` o `ene_ago`):
- `fob`, `kg`, `vu`: valor FOB, peso neto y valor unitario de todo el producto.
- `fob_base`, `kg_base`, `vu_base`: lo mismo para el producto base (café verde o cacao en grano).
- `precio_int`: promedio del precio internacional en los mismos meses (US$/kg).
- `var_*`: variación % respecto al año anterior.
- `efecto_precio`, `efecto_volumen`: descomposición del cambio del FOB respecto al año anterior (US$).
- Diversificación: `ihh`, `mercados_equivalentes`, `n_destinos`, `principal_iso3`, `principal_part`, `part_ue`.

In [7]:
precios_largo = precios.melt(["anio", "mes"], value_vars=list(PRECIO_REF.values()), var_name="serie", value_name="usd_kg")
precios_largo["producto"] = precios_largo.serie.map({v: k for k, v in PRECIO_REF.items()})


def indicadores(periodo: str) -> pd.DataFrame:
    d = PERIODOS[periodo](datos)
    sub = por_subgrupo(d)
    tot = sub.groupby(level=["producto", "anio"]).sum()
    base = sub[sub.index.get_level_values("subgrupo") == sub.index.get_level_values("producto").map(BASE)]
    base = base.droplevel("subgrupo").add_suffix("_base")
    precio = PERIODOS[periodo](precios_largo.dropna()).groupby(["producto", "anio"]).usd_kg.mean().rename("precio_int")
    out = tot.join(base).join(precio).join(diversificacion(d))
    out["vu"] = out.fob / out.kg
    out["vu_base"] = out.fob_base / out.kg_base
    efectos = []
    for producto, anio in out.index:
        if (producto, anio - 1) in out.index:
            efectos.append(descomponer(sub.loc[(producto, anio - 1)], sub.loc[(producto, anio)]).rename((producto, anio)))
    efectos = pd.DataFrame(efectos).drop(columns="cambio_fob")
    efectos.index = pd.MultiIndex.from_tuples(efectos.index, names=["producto", "anio"])
    out = out.join(efectos)
    out = out.reset_index().sort_values(["producto", "anio"])
    for col in ["fob", "kg", "vu", "vu_base", "precio_int"]:
        out[f"var_{col}"] = out.groupby("producto")[col].pct_change() * 100
    return out.assign(periodo=periodo)


ind = pd.concat([indicadores("anual"), indicadores("ene_ago")], ignore_index=True)
ind = ind[ind.anio >= ANIO_INICIO - 1]
ind.to_csv(PROCESADOS / "indicadores_anuales.csv", index=False)

vista = ind.query("periodo == 'anual' and anio >= @ANIO_INICIO").set_index(["producto", "anio"])
(vista.assign(fob=vista.fob / 1e6, kg=vista.kg / 1e6, efecto_precio=vista.efecto_precio / 1e6, efecto_volumen=vista.efecto_volumen / 1e6)
 [["fob", "kg", "vu_base", "precio_int", "efecto_precio", "efecto_volumen", "mercados_equivalentes", "part_ue"]].round(2))

C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' ke

C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' ke

fob      kg  vu_base  precio_int  efecto_precio  \
producto anio                                                        
Cacao    2015   266.64   74.40     3.25        3.14          -1.00   
         2016   294.24   80.07     3.26        2.89           6.31   
         2017   236.82   76.69     2.56        2.03         -51.66   
         2018   267.46   83.81     2.57        2.29           4.77   
         2019   295.27   90.84     2.57        2.34          -7.28   
         2020   279.54   84.73     2.72        2.37          10.64   
         2021   304.01   92.16     2.69        2.43          -3.91   
         2022   334.00  107.87     2.48        2.39         -15.80   
         2023   427.70  113.03     3.12        3.28          74.10   
         2024  1280.47  160.67     7.74        7.33         568.07   
         2025  1618.27  178.04     8.21        7.80         221.53   
Café     2015   613.68  185.15     3.31        3.53        -136.14   
         2016   762.63  241.30     3.16        3.61         -31.07   
         2017   713.90  247.26     2.89        3.32         -66.78   
         2018   682.85  261.42     2.61        2.92         -69.76   
         2019   637.30  233.02     2.73        2.88          29.42   
         2020   642.24  213.90     3.00        3.32          60.20   
         2021   769.73  195.08     3.94        4.51         192.51   
         2022  1236.12  246.67     5.01        5.63         235.68   
         2023   819.71  205.29     3.99        4.54        -230.14   
         2024  1101.75  238.52     4.62        5.62         139.09   
         2025  1801.97  242.54     7.42        8.47         675.64   

               efecto_volumen  mercados_equivalentes  part_ue  
producto anio                                                  
Cacao    2015           33.04                   5.92    65.61  
         2016           21.29                   7.12    59.46  
         2017           -5.77                   8.50    50.90  
         2018           25.87                  10.05    48.38  
         2019           35.09                   7.55    42.72  
         2020          -26.36                   8.14    41.63  
         2021           28.38                   8.88    37.73  
         2022           45.79                   8.90    28.96  
         2023           19.60                  10.16    30.14  
         2024          284.71                  10.36    34.79  
         2025          116.28                  10.40    40.26  
Café     2015            0.10                   5.86    54.44  
         2016          180.01                   5.95    51.85  
         2017           18.05                   6.88    51.15  
         2018           38.71                   6.68    50.22  
         2019          -74.97                   6.68    50.99  
         2020          -55.25                   6.92    49.64  
         2021          -65.02                   7.62    51.42  
         2022          230.70                   7.85    51.38  
         2023         -186.27                   7.33    48.19  
         2024          142.95                   6.92    45.12  
         2025           24.57                   6.05    40.37

`fob` en millones de US$, `kg` en miles de toneladas, `vu_base` y `precio_int` en US$/kg, efectos en millones de US$.

## 7. Exportaciones por región de destino

In [8]:
def por_region(periodo: str) -> pd.DataFrame:
    r = (PERIODOS[periodo](datos).groupby(["producto", "anio", "region"])[["fob_usd", "peso_neto_kg"]].sum()
         .rename(columns={"fob_usd": "fob", "peso_neto_kg": "kg"}))
    r["part_fob"] = r.fob / r.fob.groupby(level=["producto", "anio"]).transform("sum") * 100
    return r.reset_index().assign(periodo=periodo)


regiones = pd.concat([por_region("anual"), por_region("ene_ago")], ignore_index=True)
regiones.to_csv(PROCESADOS / "exportaciones_por_region.csv", index=False)
(regiones.query("periodo == 'anual' and anio in [@ANIO_INICIO, @ANIO_FIN]")
 .pivot_table(index=["producto", "region"], columns="anio", values="part_fob").round(1))

anio                                 2015  2025
producto region                                
Cacao    América Latina y el Caribe   8.1  19.6
         Asia y Medio Oriente         4.0  14.2
         Norteamérica                15.9  23.8
         Oceanía                      1.0   0.8
         Resto de Europa              5.3   1.1
         Sin país declarado           0.0   0.0
         Unión Europea               65.6  40.3
         África                       0.1   0.3
Café     América Latina y el Caribe   4.0   6.8
         Asia y Medio Oriente         6.4   4.2
         Norteamérica                30.5  43.0
         Oceanía                      1.0   1.5
         Resto de Europa              3.3   4.1
         Sin país declarado           NaN   0.0
         Unión Europea               54.4  40.4
         África                       0.4   0.0

## 8. Descomposición entre años clave

Además del cambio año a año, se descompone el cambio en ventanas que sirven para el análisis: todo el periodo (2015→2025), los
últimos dos años (2023→2025) y el año parcial (enero–agosto 2026 frente a enero–agosto 2025).

In [9]:
VENTANAS = [("anual", 2015, 2025), ("anual", 2015, 2020), ("anual", 2020, 2025), ("anual", 2023, 2025),
            ("anual", 2024, 2025), ("ene_ago", 2025, 2026)]
filas = []
for periodo, a0, a1 in VENTANAS:
    sub = por_subgrupo(PERIODOS[periodo](datos))
    for producto in PRODUCTOS:
        r = descomponer(sub.loc[(producto, a0)], sub.loc[(producto, a1)])
        filas.append({"periodo": periodo, "desde": a0, "hasta": a1, "producto": producto,
                      "fob_desde": sub.loc[(producto, a0)].fob.sum(), "fob_hasta": sub.loc[(producto, a1)].fob.sum(), **r})
ventanas = pd.DataFrame(filas)
ventanas["part_precio"] = ventanas.efecto_precio / ventanas.cambio_fob * 100
ventanas.to_csv(PROCESADOS / "descomposicion_ventanas.csv", index=False)
ventanas.assign(**{c: ventanas[c] / 1e6 for c in ["fob_desde", "fob_hasta", "cambio_fob", "efecto_precio", "efecto_volumen"]}).round(1)

C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  t = t0.join(t1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
C:\Users\Lory\AppData\Local\Temp\ipykernel_28972\3751509652.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' ke

,periodo,desde,hasta,producto,fob_desde,fob_hasta,cambio_fob,efecto_precio,efecto_volumen,part_precio
0,anual,2015,2025,Café,613.7,1802.0,1188.3,881.5,306.8,74.2
1,anual,2015,2025,Cacao,266.6,1618.3,1351.6,701.3,650.3,51.9
2,anual,2015,2020,Café,613.7,642.2,28.6,-60.4,88.9,-211.4
3,anual,2015,2020,Cacao,266.6,279.5,12.9,-36.1,49.0,-279.6
4,anual,2020,2025,Café,642.2,1802.0,1159.7,1009.6,150.1,87.1
5,anual,2020,2025,Cacao,279.5,1618.3,1338.7,798.3,540.4,59.6
6,anual,2023,2025,Café,819.7,1802.0,982.3,769.4,212.9,78.3
7,anual,2023,2025,Cacao,427.7,1618.3,1190.6,793.5,397.0,66.7
8,anual,2024,2025,Café,1101.7,1802.0,700.2,675.6,24.6,96.5
9,anual,2024,2025,Cacao,1280.5,1618.3,337.8,221.5,116.3,65.6


## 9. Serie mensual: valor unitario del producto base y precio internacional

In [10]:
base_mensual = (
    mensual[mensual.subgrupo == mensual.producto.map(BASE)]
    .groupby(["producto", "anio", "mes"])[["fob_usd", "peso_neto_kg"]].sum()
    .rename(columns={"fob_usd": "fob_base", "peso_neto_kg": "kg_base"})
)
serie = (
    precios_largo.dropna().set_index(["producto", "anio", "mes"]).usd_kg.rename("precio_int").to_frame()
    .join(base_mensual, how="outer").reset_index().query("anio >= @ANIO_INICIO")
    .sort_values(["producto", "anio", "mes"])
)
serie["vu_base"] = serie.fob_base / serie.kg_base
serie.to_csv(PROCESADOS / "precio_y_valor_unitario_mensual.csv", index=False)
serie.tail(4)

,producto,anio,mes,precio_int,fob_base,kg_base,vu_base
1598,Café,2026,6,6.79,9.475743e+07,1.526641e+07,6.206921
1599,Café,2026,7,7.91,1.511430e+08,2.314941e+07,6.529021
1600,Café,2026,8,7.97,2.191499e+08,2.990731e+07,7.327637
1601,Café,2026,9,7.26,NaN,NaN,NaN


## 10. Datos para el dashboard (`web/data/datos.json`)

Tablas compactas (listas en vez de objetos) para que la página cargue rápido:
- `pais`: `[año, producto, iso3, fob, kg, fob_p, kg_p]`, donde `_p` es el acumulado enero–último mes.
- `ind`: indicadores por producto y año (objetos), para `anual` y `ene_ago`.
- `region`: `[periodo, producto, año, región, fob, kg]`.
- `mensual`: `["AAAA-MM", producto, valor unitario base, kg base, precio internacional]`.

In [11]:
def num(x, dec: int = 1):
    return None if pd.isna(x) else round(float(x), dec)


datos["fob_p"] = datos.fob_usd.where(datos.mes <= ULTIMO_MES, 0)
datos["kg_p"] = datos.peso_neto_kg.where(datos.mes <= ULTIMO_MES, 0)
pais_anual = (datos[datos.anio >= ANIO_INICIO - 1]
              .groupby(["anio", "producto", "iso3"], as_index=False)[["fob_usd", "peso_neto_kg", "fob_p", "kg_p"]].sum())
paises_json = {
    r.iso3: [r.pais, ORDEN_REGIONES.index(r.region)]
    for r in tabla_paises.drop_duplicates("iso3").itertuples() if r.iso3 in set(pais_anual.iso3)
}
COLUMNAS_IND = {
    "fob": 1, "kg": 1, "vu": 3, "vu_base": 3, "precio_int": 3, "efecto_precio": 1, "efecto_volumen": 1,
    "var_fob": 1, "var_kg": 1, "var_vu": 1, "var_vu_base": 1, "var_precio_int": 1,
    "ihh": 0, "mercados_equivalentes": 2, "n_destinos": 0, "principal_part": 1, "part_ue": 1,
}
web = {
    "meta": {
        "actualizado_sunat": metadatos["G8"]["fecha_descarga"],
        "actualizado_precios": metadatos["PINK"]["actualizado_por_bm"],
        "anio_inicio": ANIO_INICIO,
        "anio_fin": ANIO_FIN,
        "anio_parcial": ANIO_PARCIAL,
        "ultimo_mes": ULTIMO_MES,
        "ultimo_mes_precio": f"{int(ultimo_precio.anio)}-{int(ultimo_precio.mes):02d}",
    },
    "productos": PRODUCTOS,
    "base": [BASE[p] for p in PRODUCTOS],
    "precio_ref": ["Café arábica (OIC, otros suaves)", "Cacao (ICCO)"],
    "regiones": ORDEN_REGIONES,
    "paises": paises_json,
    "pais": [
        [int(r.anio), PRODUCTOS.index(r.producto), r.iso3, num(r.fob_usd), num(r.peso_neto_kg), num(r.fob_p), num(r.kg_p)]
        for r in pais_anual.itertuples()
    ],
    "ind": [
        {"periodo": r["periodo"], "p": PRODUCTOS.index(r["producto"]), "anio": int(r["anio"]), "principal": r["principal_iso3"],
         **{c: num(r[c], dec) for c, dec in COLUMNAS_IND.items()}}
        for _, r in ind[ind.anio >= ANIO_INICIO].iterrows()
    ],
    "region": [
        [r.periodo, PRODUCTOS.index(r.producto), int(r.anio), ORDEN_REGIONES.index(r.region), num(r.fob), num(r.kg)]
        for r in regiones[regiones.anio >= ANIO_INICIO].itertuples()
    ],
    "mensual": [
        [f"{int(r.anio)}-{int(r.mes):02d}", PRODUCTOS.index(r.producto), num(r.vu_base, 3), num(r.kg_base, 0), num(r.precio_int, 3)]
        for r in serie.itertuples()
    ],
}
ruta = WEB_DATA / "datos.json"
ruta.write_text(json.dumps(web, ensure_ascii=False, separators=(",", ":"), allow_nan=False), encoding="utf-8")
print(f"{ruta.relative_to(RAIZ)}: {ruta.stat().st_size / 1e6:.2f} MB · {len(web['pais']):,} filas país-año · {len(web['ind'])} filas de indicadores")

web\data\datos.json: 0.12 MB · 1,578 filas país-año · 46 filas de indicadores


### Verificación: los totales del JSON cuadran con los indicadores

In [12]:
chk = pais_anual[pais_anual.anio < ANIO_PARCIAL].groupby(["producto", "anio"]).fob_usd.sum()
ref = ind.query("periodo == 'anual'").set_index(["producto", "anio"]).fob
assert (chk.loc[ref.index] - ref).abs().max() < 1, "Los totales del dashboard no cuadran"
chk_region = regiones.query("periodo == 'anual'").groupby(["producto", "anio"]).fob.sum()
assert (chk_region.loc[ref.index] - ref).abs().max() < 1, "La suma por región no cuadra"
print("OK: el FOB del dashboard y la suma por región coinciden con los indicadores anuales")

OK: el FOB del dashboard y la suma por región coinciden con los indicadores anuales
